In [14]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.model_selection import GroupKFold, GridSearchCV

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings("ignore")

In [3]:
data = pd.read_csv("C:/Users/Durosimi/PROJECTS/Parkinson/data/processed/featured_data.csv")
data.head()

,subject,age,sex,test_time,motor_updrs,total_updrs,jitter,jitter_abs,jitter_rap,jitter_ppq5,...,shimmer_db,shimmer_apq3,shimmer_apq5,shimmer_apq11,shimmer_dda,nhr,hnr,rpde,dfa,ppe
0,1,72,False,5.6431,28.199,34.398,0.00662,0.000034,0.00401,0.00317,...,0.230,0.01438,0.01309,0.01662,0.04314,0.014290,21.640,0.41888,0.54842,0.16006
1,1,72,False,12.6660,28.447,34.894,0.00300,0.000017,0.00132,0.00150,...,0.179,0.00994,0.01072,0.01689,0.02982,0.011112,27.183,0.43493,0.56477,0.10810
2,1,72,False,19.6810,28.695,35.389,0.00481,0.000025,0.00205,0.00208,...,0.181,0.00734,0.00844,0.01458,0.02202,0.020220,23.047,0.46222,0.54405,0.21014
3,1,72,False,25.6470,28.905,35.810,0.00528,0.000027,0.00191,0.00264,...,0.327,0.01106,0.01265,0.01963,0.03317,0.027837,24.445,0.48730,0.57794,0.33277
4,1,72,False,33.6420,29.187,36.375,0.00335,0.000020,0.00093,0.00130,...,0.176,0.00679,0.00929,0.01819,0.02036,0.011625,26.126,0.47188,0.56122,0.19361


In [5]:
target = 'motor_updrs'
features = [
     "jitter",
    "shimmer",
    "hnr",
    "rpde",
    "ppe",
    "sex"
]

In [7]:
X = data[features]
y = data[target]

In [11]:
groups = data["subject"]

In [16]:
group_fold = GroupKFold(n_splits=5)

### This ensures that a patient appears only in train or validation
- No leakage

### Random Forest Grouping

In [25]:
rf_pipeline = Pipeline([
    ("scaler", StandardScaler()), 
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

rf_param_grid = {
    "model__n_estimators": [300, 500],
    "model__max_depth": [8, 12, 16],
    "model__min_samples_leaf": [3, 5, 10]
}

rf_search = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv=group_fold,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X, y, groups=groups)

rf_search.best_params_


Fitting 5 folds for each of 18 candidates, totalling 90 fits


{'model__max_depth': 8,
 'model__min_samples_leaf': 10,
 'model__n_estimators': 300}

### SVR Group Tuning

In [28]:
svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVR())
])

svr_param_grid = {
    "model__C": [1, 10, 50],
    "model__epsilon": [0.05, 0.1, 0.2],
    "model__gamma": ["scale", 0.01, 0.05]
}

svr_search = GridSearchCV(
    svr_pipeline,
    svr_param_grid,
    cv=group_fold,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

svr_search.fit(X, y, groups=groups)

svr_search.best_params_


Fitting 5 folds for each of 27 candidates, totalling 135 fits


{'model__C': 1, 'model__epsilon': 0.05, 'model__gamma': 0.01}

### COMPARING CROSS-VALIDATED PERFORMANCE

In [33]:
results = pd.DataFrame([
    {
        "model": "Random Forest",
        "CV_MAE": -rf_search.best_score_
    },
    {
        "model": "SVR",
        "CV_MAE": -svr_search.best_score_
    }
])

results

,model,CV_MAE
0,Random Forest,7.571333
1,SVR,7.626510


In [36]:
best_model = rf_search if rf_search.best_score_ > svr_search.best_score_ else svr_search

final_model = best_model.best_estimator_

final_model


Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 RandomForestRegressor(max_depth=8, min_samples_leaf=10,
                                       n_estimators=300, n_jobs=-1,
                                       random_state=42))])

In [41]:
import joblib

joblib.dump(
    final_model,
    "C:/Users/Durosimi/PROJECTS/Parkinson/models/final_motor_updrs_groupcv.pkl"
)

print("✅ Final GroupCV model saved")

✅ Final GroupCV model saved


In [44]:
sample = X.sample(5, random_state=42)
preds = final_model.predict(sample)

pd.DataFrame({
    "Predicted_motor_UPDRS": preds
})


,Predicted_motor_UPDRS
0,21.815061
1,18.055014
2,18.433907
3,22.961409
4,23.962112


### Using subject-level GroupKFold cross-validation, the model achieves a mean absolute error of approximately 7.6 points on the motor UPDRS score, which is consistent with prior work using voice-only telemonitoring features.